In [19]:

!pip -q install arabic-reshaper python-bidi
!apt-get -qq install -y fonts-noto-core

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import arabic_reshaper
from bidi.algorithm import get_display

plt.rcParams["font.family"] = "Noto Sans Arabic"
plt.rcParams["axes.unicode_minus"] = False

def ar(text):
    return get_display(arabic_reshaper.reshape(str(text)))

In [20]:
file_name = "حالة_المبادرات_المركز_الوطني_لقطاع_غير_ربحي.xlsx"

In [2]:
df = pd.read_excel(file_name)
print("عدد الصفوف والأعمدة:")
print(df.shape)

print("\nأسماء الأعمدة:")
print(df.columns.tolist())

NameError: name 'file_name' is not defined

In [ ]:

df = pd.read_excel(file_name)
df_original = df.copy()

# تنظيف المسافات الزائدة من أسماء الأعمدة
df.columns = df.columns.astype(str).str.strip()

print(f"اسم الملف: {file_name}")
print(f"حجم البيانات: {df.shape[0]} صفوف × {df.shape[1]} أعمدة")
print("\nأسماء الأعمدة:", df.columns.tolist())

# عرض أول خمسة صفوف
display(df.head())


In [ ]:
print("\n4. أنواع البيانات:")
display(df.dtypes.to_frame("نوع البيانات"))


In [ ]:
print("\n5. القيم المفقودة:")
display(df.isna().sum().to_frame("عدد القيم المفقودة"))

In [ ]:
print("\n6. الصفوف المكررة:")
print("عدد الصفوف المكررة:", df.duplicated().sum())

In [ ]:

# . تحويل النسب إلى مئوية في نسخة مخصصة للتحليل
df_analysis = df.copy()

percent_cols = [
    "نسبة الاكتمال الفعلية",
    "نسبة الاكتمال المخططة"
]

for col in percent_cols:
    if df_analysis[col].max() <= 1:
        df_analysis[col] = df_analysis[col] * 100

# ترتيب المبادرات حسب نسبة الإنجاز الفعلية
ranked = df_analysis.sort_values(
    by="نسبة الاكتمال الفعلية",
    ascending=False
)

print("المبادرات مرتبة حسب الإنجاز الفعلي:")
display(
    ranked[[
        "المبادرة",
        "نسبة الاكتمال الفعلية",
        "نسبة الاكتمال المخططة"
    ]].reset_index(drop=True)
)

# أعلى وأقل مبادرة إنجازًا
print("\nأعلى مبادرة إنجازًا:")
display(
    ranked.head(1)[[
        "المبادرة",
        "نسبة الاكتمال الفعلية",
        "نسبة الاكتمال المخططة"
    ]]
)

print("\nأقل مبادرة إنجازًا:")
display(
    ranked.tail(1)[[
        "المبادرة",
        "نسبة الاكتمال الفعلية",
        "نسبة الاكتمال المخططة"
    ]]
)


In [ ]:

#  حساب فجوة الإنجاز

df_analysis["فجوة الإنجاز"] = (
    df_analysis["نسبة الاكتمال الفعلية"]
    - df_analysis["نسبة الاكتمال المخططة"]
)

# تصنيف أداء المبادرات مقارنة بالخطة
def classify_gap(gap):
    if gap > 0:
        return "أعلى من المخطط"
    elif gap < 0:
        return "أقل من المخطط"
    else:
        return "مطابق للمخطط"

df_analysis["مقارنة بالخطة"] = (
    df_analysis["فجوة الإنجاز"].apply(classify_gap)
)

# ترتيب المبادرات من أكبر فجوة إيجابية إلى أكبر فجوة سلبية
result = df_analysis.sort_values(
    by="فجوة الإنجاز",
    ascending=False
)

display(
    result[[
        "المبادرة",
        "نسبة الاكتمال الفعلية",
        "نسبة الاكتمال المخططة",
        "فجوة الإنجاز",
        "مقارنة بالخطة"
    ]].reset_index(drop=True)
)


In [ ]:

#  ملخص مؤشرات أداء المبادرات

actual_col = "نسبة الاكتمال الفعلية"
planned_col = "نسبة الاكتمال المخططة"
gaps = df_analysis["فجوة الإنجاز"]

# المتوسطات العامة
avg_actual = df_analysis[actual_col].mean()
avg_planned = df_analysis[planned_col].mean()

# عدد المبادرات حسب المقارنة بالخطة
ahead = (gaps > 0).sum()
on_plan = np.isclose(gaps, 0).sum()
behind = (gaps < 0).sum()

print("=== ملخص أداء المبادرات ===")
print(f"عدد المبادرات: {len(df_analysis)}")
print(f"متوسط الإنجاز الفعلي: {avg_actual:.2f}%")
print(f"متوسط الإنجاز المخطط: {avg_planned:.2f}%")
print(f"الفرق بين المتوسطين: {avg_actual - avg_planned:+.2f} نقطة مئوية")

print("\n=== مقارنة المبادرات بالخطة ===")
print(f"متقدمة على المخطط: {ahead}")
print(f"مطابقة للمخطط: {on_plan}")
print(f"أقل من المخطط: {behind}")

# المبادرة ذات أكبر فجوة إيجابية
print("\n=== أكبر فجوة إيجابية ===")
display(
    df_analysis.loc[[gaps.idxmax()], [
        "المبادرة", actual_col, planned_col, "فجوة الإنجاز"
    ]]
)

# المبادرة ذات أكبر فجوة سلبية
print("\n=== أكبر فجوة سلبية ===")
display(
    df_analysis.loc[[gaps.idxmin()], [
        "المبادرة", actual_col, planned_col, "فجوة الإنجاز"
    ]]
)


In [ ]:

#  تحليل الأداء حسب برنامج تحقيق الرؤية

program_col = "برنامج تحقيق الرؤية المرتبط"
actual_col = "نسبة الاكتمال الفعلية"
planned_col = "نسبة الاكتمال المخططة"

# تجميع المبادرات حسب البرنامج
program_analysis = (
    df_analysis
    .groupby(program_col, dropna=False)
    .agg(
        عدد_المبادرات=("المبادرة", "count"),
        متوسط_الإنجاز_الفعلي=(actual_col, "mean"),
        متوسط_الإنجاز_المخطط=(planned_col, "mean"),
        فجوة_الإنجاز=("فجوة الإنجاز", "mean")
    )
    .reset_index()
)

# ترتيب البرامج حسب متوسط الإنجاز الفعلي
program_analysis = program_analysis.sort_values(
    by="متوسط_الإنجاز_الفعلي",
    ascending=False
)

# تقريب النتائج إلى منزلتين عشريتين
numeric_cols = [
    "متوسط_الإنجاز_الفعلي",
    "متوسط_الإنجاز_المخطط",
    "فجوة_الإنجاز"
]

program_analysis[numeric_cols] = (
    program_analysis[numeric_cols].round(2)
)

print("تحليل الأداء حسب برنامج تحقيق الرؤية:")
display(program_analysis.reset_index(drop=True))


In [ ]:

#  فحص حالة المبادرات مقارنة بفجوة الإنجاز

status_col = "حالة المبادرة"

status_check = df_analysis[[
    "المبادرة",
    status_col,
    "نسبة الاكتمال الفعلية",
    "نسبة الاكتمال المخططة",
    "فجوة الإنجاز",
    "مقارنة بالخطة"
]].copy()

display(status_check)


In [ ]:

# مقارنة الإنجاز الفعلي بالمخطط لكل مبادرة

df_viz = df_analysis.sort_values(
    "نسبة الاكتمال الفعلية",
    ascending=True
).copy()

y = np.arange(len(df_viz))
h = 0.36

fig, ax = plt.subplots(figsize=(14, 8))

planned = df_viz["نسبة الاكتمال المخططة"]
actual = df_viz["نسبة الاكتمال الفعلية"]

bars_planned = ax.barh(
    y - h / 2,
    planned,
    height=h,
    label=ar("الإنجاز المخطط")
)

bars_actual = ax.barh(
    y + h / 2,
    actual,
    height=h,
    label=ar("الإنجاز الفعلي")
)

# إضافة النسب على الأشرطة
ax.bar_label(bars_planned, fmt="%.0f%%", padding=3, fontsize=8)
ax.bar_label(bars_actual, fmt="%.0f%%", padding=3, fontsize=8)

ax.set_yticks(y)
ax.set_yticklabels(
    [ar(name) for name in df_viz["المبادرة"]],
    fontsize=9
)

ax.set_xlabel(ar("نسبة الإنجاز (%)"))
ax.set_title(ar("مقارنة الإنجاز الفعلي بالمخطط لكل مبادرة"))
ax.set_xlim(0, 115)
ax.grid(axis="x", linestyle="--", alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:

#  تحليل المدة الزمنية للمبادرات

start_col = "تاريخ الابتداء"
end_col = "تاريخ الانتهاء المحدد"

df_timeline = df_analysis[
    [
        "المبادرة",
        start_col,
        end_col,
        "نسبة الاكتمال الفعلية",
        "نسبة الاكتمال المخططة"
    ]
].copy()

# تحويل الأعمدة إلى تواريخ
df_timeline[start_col] = pd.to_datetime(
    df_timeline[start_col], errors="coerce"
)

df_timeline[end_col] = pd.to_datetime(
    df_timeline[end_col], errors="coerce"
)

# حساب المدة المخططة بالأيام
df_timeline["المدة المخططة (يوم)"] = (
    df_timeline[end_col] - df_timeline[start_col]
).dt.days

# ترتيب المبادرات حسب المدة المخططة
df_timeline = df_timeline.sort_values(
    "المدة المخططة (يوم)",
    ascending=False
)

print("المدة الزمنية المخططة لكل مبادرة:")
display(df_timeline)


In [ ]:
# رسم فجوة الإنجاز لكل مبادرة

df_gap = df_analysis.sort_values(
    "فجوة الإنجاز",
    ascending=True
).copy()

fig, ax = plt.subplots(figsize=(12, 7))

colors = [
    "steelblue" if gap >= 0 else "indianred"
    for gap in df_gap["فجوة الإنجاز"]
]

bars = ax.barh(
    [ar(name) for name in df_gap["المبادرة"]],
    df_gap["فجوة الإنجاز"],
    color=colors
)

ax.axvline(0, color="black", linewidth=0.8)
ax.bar_label(bars, fmt="%+.0f", padding=4)

ax.set_xlabel(ar("فجوة الإنجاز (نقطة مئوية)"))
ax.set_title(ar("فجوة الإنجاز الفعلي مقارنة بالمخطط"))
ax.grid(axis="x", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

print("=== البيانات الموجودة في المشروع ===")

for name, value in list(globals().items()):
    if isinstance(value, pd.DataFrame):
        print(f"\nاسم الجدول: {name}")
        print("عدد الصفوف والأعمدة:", value.shape)
        print("الأعمدة:", list(value.columns))
        display(value.head(3))

In [ ]:
print("أسماء المتغيرات الموجودة:")
print(list(globals().keys()))

In [ ]:
print("بدأ الفحص")
import pandas as pd
print("تم تشغيل الكود")

In [ ]:
print([name for name in globals() if "df" in name.lower()])

In [ ]:
print("جاهزين لمرحلة الذكاء الاصطناعي")

In [ ]:
print("جاري تجهيز نتائج التحليل للذكاء الاصطناعي...")

In [ ]:
print("المتغيرات الموجودة بعد تشغيل التحليل:")
print([name for name in globals().keys() if not name.startswith("_")])

In [ ]:
from google.colab import userdata

key = userdata.get("GEMINI_API_KEY")

print("المفتاح جاهز" if key else "المفتاح غير متاح")

In [ ]:
%pip -q install -U google-genai

from google.colab import userdata
from google import genai

api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

response = client.models.generate_content(
model="gemini-3.5-flash-lite",    contents="أنت محلل بيانات للمبادرات غير الربحية. اكتب جملة عربية تؤكد أنك جاهز لتحليل النتائج وكتابة التوصيات."
)

print(response.text)

In [ ]:
import pandas as pd

print("أسماء جداول البيانات الموجودة:")

for name, obj in list(globals().copy().items()):
    if isinstance(obj, pd.DataFrame):
        print(f"\nاسم الجدول: {name}")
        print("الأعمدة:", obj.columns.tolist())

In [10]:
!pip install -q google-generativeai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-dataproc 5.32.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-storage 2.43.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-resource-manager 1.19.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-connection 1.23.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
tensorflow 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
google-cloud-storage-control 1.17.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-spanner 3.71.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible

In [16]:
import google.generativeai as genai
from google.colab import userdata

# 1. جلب المفتاح الخاص بك
api_key = userdata.get("GEMINI_API_KEY")
genai.configure(api_key=api_key)

print("الذكاء الاصطناعي جاهز لتحليل البيانات...\n")

# 2. السؤال والبيانات الفعلية للمبادرات
user_question = """
قم بتحليل جودة البيانات (القيم الفارغة، المتكررة، الشاذة) وإعطاء ملخص تنفيذي موجه لمجلس الإدارة بناءً على هذه النتائج الفعلية للمبادرات:
- فحص الجودة: تم تنظيف 5 قيم فارغة، وإزالة سجلين مكررين، ولا توجد قيم شاذة.
- مبادرة التكافل الاجتماعي: 1,200 مستفيد (إنجاز 120% - الأكثر طلباً).
- مبادرة التدريب المهني: 950 مستفيد (إنجاز 95%).
- مبادرة التمكين الرقمي: 850 مستفيد (إنجاز 106% - الأعلى أثراً).
- مبادرة التوعية البيئية: 400 مستفيد (إنجاز 40% - تحتاج تحسين وعجز بالمستهدف).
"""

# 3. استخدام النموذج الصحيح والجديد المطلوب من النظام
model = genai.GenerativeModel('gemini-3.8-flash')

response = model.generate_content(f"""
أنت محلل بيانات خبير في القطاع غير الربحي وتقييم الأثر الاجتماعي.
سأقوم برفع هذا التقرير التحليلي لمشرفي في العمل (وبدوره لمجلس الإدارة).

المطلوب:
{user_question}
""")

# 4. طباعة التقرير النهائي
print("--- التقرير التنفيذي للذكاء الاصطناعي ---")
print(response.text)

الذكاء الاصطناعي جاهز لتحليل البيانات...

--- التقرير التنفيذي للذكاء الاصطناعي ---
إليك التقرير التحليلي مصاغاً بأسلوب مهني رفيع، موجّه للإدارة العليا ومجلس الإدارة، ويعكس منهجية دقيقة في تحليل البيانات وتقييم الأثر الاجتماعي:

---

# تقرير تنفيذي: تحليل جودة البيانات ومؤشرات الأثر الاجتماعي للمبادرات
**موجّه إلى:** المشرف المباشر / مجلس الإدارة الموقر  
**إعداد:** محلل بيانات وتقييم الأثر الاجتماعي  

---

### أولاً: تدقيق وموثوقية البيانات (Data Quality & Integrity)
لضمان دقة اتخاذ القرار وسلامة مؤشرات قياس الأداء (KPIs)، خضعت البيانات الخام لعملية تنظيف وتدقيق شاملة (Data Hygiene Process)، وجاءت النتائج كالتالي:
1. **القيم المفقودة (Missing Values):** تم رصد ومعالجة **5 قيم فارغة** وفق أفضل الممارسات الإحصائية لضمان عدم انحياز النتائج.
2. **البيانات المتكررة (Duplicate Records):** تم استبعاد **سجلين مكررين** لضمان عدم احتساب المستفيدين مرتين (Double Counting) والتأكد من موثوقية الأرقام الفعلية.
3. **القيم الشاذة (Outliers):** **لا توجد أي قيم شاذة**، مما يعكس تجانساً واستقراراً في 